# **Stratified Sampling**

The paper Detecting Social Bots on Facebook in an Information Veracity Context describes a log-based binning approach for stratified sampling. They first partitioned users into bins of size ten based on comment volume, then adjusted for even representation by using 100 log-spaced bins to ensure a balanced distribution across different levels of comment activity​.

*Data*

In [ ]:
#LIBRARIES
import json
import pandas as pd
import numpy as np
import openpyxl
import os
import re

In [ ]:
with open("/content/Facebook_Comments (Merged).json", "r", encoding="utf-8") as f:
    data = json.load(f)

*Data Analytics (know comment counts of user)*

In [ ]:
# Comments per user
user_comment_counts = {entry["userID"]: len(entry["comments"]) for entry in data}

In [ ]:
# DataFrame
df = pd.DataFrame(list(user_comment_counts.items()), columns=["userID", "comment_count"])

In [ ]:
df.to_excel("Comment Counts (Whole Data).xlsx", index=False, engine="openpyxl")
print("Saved comment counts per user to 'comment_counts.xlsx'.")

Saved comment counts per user to 'comment_counts.xlsx'.


In [ ]:
# min and max
min_comments = df["comment_count"].min()
max_comments = df["comment_count"].max()
# comment counts (sorted)
unique_comment_counts = sorted(df["comment_count"].unique())

In [ ]:
print(f"Least number of comments: {min_comments}")
print(f"Highest number of comments: {max_comments}")
print(f"Unique comment counts (sorted): {unique_comment_counts}")

Least number of comments: 1
Highest number of comments: 423
Unique comment counts (sorted): [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int

*Log-based Binning*

In [ ]:
df["log_comment_count"] = np.log10(df["comment_count"] + 1)  # Avoid log(0)

In [ ]:
# log-spaced bins between 1 and max comment count (423)
num_bins = 10
bins = np.logspace(np.log10(1), np.log10(df["comment_count"].max()), num=num_bins)

9 bins instead of 10 because pd.cut() assigns values into bins based on intervals, and the way np.logspace() is generating edges means that the highest value (423) falls within the 9th bin, not forming a 10th bin.

adjust bin (i.e., 11) if needed

In [ ]:
# USERS TO BINS
df["bins"] = pd.cut(df["comment_count"], bins=bins, labels=False, include_lowest=True)

In [ ]:
print("Log-based Bins (comment count ranges):")
for i in range(len(bins) - 1):
    print(f"Bin {i + 1}: {bins[i]:.2f} to {bins[i+1]:.2f}")

Log-based Bins (comment count ranges):
Bin 1: 1.00 to 1.96
Bin 2: 1.96 to 3.83
Bin 3: 3.83 to 7.51
Bin 4: 7.51 to 14.70
Bin 5: 14.70 to 28.78
Bin 6: 28.78 to 56.35
Bin 7: 56.35 to 110.33
Bin 8: 110.33 to 216.04
Bin 9: 216.04 to 423.00


In [ ]:
bin_counts = df["bins"].value_counts().sort_index()
print("\nUser Distribution Per Bin:")
print(bin_counts)


User Distribution Per Bin:
bins
0.0    205374
1.0     79722
2.0     28848
3.0      8999
4.0      3112
5.0       901
6.0       225
7.0        54
8.0        10
Name: count, dtype: int64


In [ ]:
# total comments per bin
df["total_comments"] = df["comment_count"]
bin_comment_counts = df.groupby("bins")["total_comments"].sum().sort_index()

In [ ]:
print("\nTotal Comments Per Bin:")
print(bin_comment_counts)


Total Comments Per Bin:
bins
0.0    205374
1.0    183234
2.0    143497
3.0     90520
4.0     60718
5.0     34061
6.0     16286
7.0      7972
8.0      2646
Name: total_comments, dtype: int64


In [ ]:
# remove this if data is ample enough

# PROPORTIONAL SAMPLING: Adjust sample sizes per bin
# total_users = 1000  # Target sample size
# bin_weights = bin_counts / bin_counts.sum()  # Proportions
# bin_samples = (bin_weights * total_users).astype(int)  # Adjusted sample sizes

In [ ]:
# Sample users per bin
# sampled_users = df.groupby("bins", group_keys=False).apply(lambda x: x.sample(n=min(len(x), bin_samples.get(x.name, 0)), random_state=42))

In [ ]:
# STRATIFIED SAMPLING
sampled_users = df.groupby("bins", group_keys=False).apply(lambda x: x.sample(n=min(len(x), int(1000/(num_bins-1))), random_state=42))
print(f"Total Users in Sampled Data: {sampled_users.shape[0]}")

Total Users in Sampled Data: 841


<ipython-input-17-907994b06fe8>:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled_users = df.groupby("bins", group_keys=False).apply(lambda x: x.sample(n=min(len(x), int(1000/(num_bins-1))), random_state=42))


some quantity i tried: 764, 841, 995

In [ ]:
# GET SAMPLED USER IDs
sampled_user_ids = set(sampled_users["userID"])

In [ ]:
# FILTER JSON TO INCLUDE ONLY SAMPLED USERS
filtered_data = [entry for entry in data if entry["userID"] in sampled_user_ids]

In [ ]:
# Function to clean illegal characters
def clean_text(text):
    if isinstance(text, str):  # Ensure text is a string
        text = re.sub(r"[\x00-\x1F\x7F-\x9F]", " ", text)  # Remove control characters
        return text.strip()  # Remove leading/trailing spaces
    return text

In [ ]:
# FLATTEN JSON STRUCTURE
flattened_data = []
for user in filtered_data:
    user_id = user["userID"]
    label = user.get("label", None)
    features = user.get("features", [])

    for comment in user["comments"]:
        flattened_data.append({
            "userID": user_id,
            "facebookPost": clean_text(comment["facebookPost"]),
            "newsOutlet": clean_text(comment["newsOutlet"]),
            "postDate": clean_text(comment["postDate"]),
            "postTime": clean_text(comment["postTime"]),
            "commentID": clean_text(comment["commentID"]),
            "commentText": clean_text(comment["commentText"]),
            "commentDate": clean_text(comment["commentDate"]),
            "commentTime": clean_text(comment["commentTime"]),
            "attachment": clean_text(comment["attachment"]),
            "isReply": clean_text(comment["isReply"]),
            "parentCommentID": clean_text(comment["parentCommentID"]),
            "depth": clean_text(comment["depth"]),
            "likes": clean_text(comment["likes"]),
            "label": clean_text(label),
            "features": clean_text(", ".join(map(str, features)))  # Convert list to string
        })

In [ ]:
# DATAFRAME FROM FLATTENED DATA
flattened_df = pd.DataFrame(flattened_data)

In [ ]:
# number of rows
print(f"Total Comments Exported: {flattened_df.shape[0]}")

Total Comments Exported: 27174


In [ ]:
# EXPORT
exp_stf = "Stratified Sample.xlsx"
flattened_df.to_excel(exp_stf, index=False, engine="openpyxl")

exports all comments of the sampled users present in the dataset

In [ ]:
if os.path.exists(exp_stf):
    print(f"File '{exp_stf}' successfully saved!")
else:
    print("Error: File was not saved. Check write permissions or path.")

File 'Stratified Sample.xlsx' successfully saved!
